In [17]:
"""
Combine MindBigData2023 train.csv + test.csv into per-date .npz files.

Two phases:
  1. Stream both CSVs in chunks → append matching rows to per-date temp CSVs
  2. For each temp CSV → convert to compressed .npz and delete temp file

Memory usage at any time: ~one chunk (1024 rows) during phase 1,
~one date's rows during phase 2.

Output: data/interim/by_date/{date}.npz

Each .npz contains:
  - label:  array (n,)
  - meta:   structured array with label_source, label_pos,
            timestamp, sessionnum, blocknum, blockpos
  - eeg:    float32 array (n, 128, 500)  — scaled to µV (ADC ÷ 1000)
  - images: float32 array (n, 28, 28)

Usage:
    python scripts/data/preprocess/raw2interim/chunk_by_date.py
"""

"\nCombine MindBigData2023 train.csv + test.csv into per-date .npz files.\n\nTwo phases:\n  1. Stream both CSVs in chunks → append matching rows to per-date temp CSVs\n  2. For each temp CSV → convert to compressed .npz and delete temp file\n\nMemory usage at any time: ~one chunk (1024 rows) during phase 1,\n~one date's rows during phase 2.\n\nOutput: data/interim/by_date/{date}.npz\n\nEach .npz contains:\n  - label:  array (n,)\n  - meta:   structured array with label_source, label_pos,\n            timestamp, sessionnum, blocknum, blockpos\n  - eeg:    float32 array (n, 128, 500)  — scaled to µV (ADC ÷ 1000)\n  - images: float32 array (n, 28, 28)\n\nUsage:\n    python scripts/data/preprocess/raw2interim/chunk_by_date.py\n"

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd

In [19]:
PROJECT_ROOT = Path.cwd().resolve().parents[0]   # keep as Path, not str
RAW_DIR  = PROJECT_ROOT / "data" / "raw"
OUT_DIR  = PROJECT_ROOT / "notebooks" / "workspace" /"by_date"
TEMP_DIR = PROJECT_ROOT / "notebooks" / "workspace" / "by_date_tmp"

N_CHANNELS = 128
N_SAMPLES  = 500
IMG_PIXELS = 784
CHUNKSIZE  = 16  # quick test: process only first N chunks per CSV; set None for full run

LABEL_COL = "label"
META_COLS = ["label_source", "label_pos", "timestamp", "sessionnum", "blocknum", "blockpos"]
IMG_COLS  = [f"label_imgpix_{i}" for i in range(IMG_PIXELS)]

In [20]:
def get_eeg_cols(columns):
    excluded = {LABEL_COL, *META_COLS, *IMG_COLS}
    return [c for c in columns if c not in excluded]

def to_date(ts_series):
    ts = pd.to_numeric(ts_series, errors="coerce")
    unit = "ms" if ts.median() > 1e12 else "s"
    return pd.to_datetime(ts, unit=unit, utc=True).dt.strftime("%Y-%m-%d")

In [21]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

csv_paths = [p for p in [RAW_DIR / "train.csv", RAW_DIR / "test.csv"] if p.exists()]

# Read only column names (fast, no data rows), then pick EEG columns.
header_cols = pd.read_csv(csv_paths[0], nrows=0).columns.tolist()
eeg_cols = get_eeg_cols(header_cols)

In [22]:
# Phase 1: stream CSVs, append rows to per-date temp CSVs
MAX_CHUNKS = 2

for p in csv_paths:
    print(f"Phase 1: streaming {p.name}...")
    n_rows = 0
    for chunk_idx, chunk in enumerate(pd.read_csv(p, chunksize=CHUNKSIZE, low_memory=False), start=1):
        dates = to_date(chunk["timestamp"])
        for date, group in chunk.groupby(dates):
            temp_file = TEMP_DIR / f"{date}.csv"
            group.to_csv(temp_file, mode="a", header=not temp_file.exists(), index=False)
        n_rows += len(chunk)
        print(f"  {n_rows:,} rows processed...", end="\r")

        if MAX_CHUNKS is not None and chunk_idx >= MAX_CHUNKS:
            print(f"  stopped early at {chunk_idx} chunks (test mode).")
            break
    print(f"  {n_rows:,} rows processed.   ")


Phase 1: streaming train.csv...
  stopped early at 2 chunks (test mode).
  32 rows processed.   
Phase 1: streaming test.csv...
  stopped early at 2 chunks (test mode).
  32 rows processed.   


In [23]:
# Phase 2: convert each temp CSV to .npz
temp_files = sorted(TEMP_DIR.glob("*.csv"))
print(f"\nPhase 2: saving {len(temp_files)} date(s)...")
for i, temp_file in enumerate(temp_files, 1):
    date = temp_file.stem
    df = pd.read_csv(temp_file, low_memory=False)

    label  = df[LABEL_COL].to_numpy()
    meta   = df[META_COLS].reset_index(drop=True)
    images = df[IMG_COLS].to_numpy(dtype=np.float32).reshape(-1, 28, 28)
    eeg    = df[eeg_cols].to_numpy(dtype=np.float32).reshape(-1, N_CHANNELS, N_SAMPLES) / 1000.0

    out_path = OUT_DIR / f"{date}.npz"
    np.savez_compressed(out_path, label=label, meta=meta.to_records(index=False), images=images, eeg=eeg)
    temp_file.unlink()
    print(f"  [{i}/{len(temp_files)}] {date}: {len(meta):>6} rows → {out_path.name}")


Phase 2: saving 2 date(s)...
  [1/2] 2023-01-24:     32 rows → 2023-01-24.npz
  [2/2] 2023-05-23:     32 rows → 2023-05-23.npz


In [24]:
TEMP_DIR.rmdir()
print(f"\nDone. Output: {OUT_DIR}")


Done. Output: /workspace/notebooks/workspace/by_date


In [30]:
npz_path = sorted(OUT_DIR.glob("*.npz"))[0]

A = np.load(npz_path, allow_pickle=True)
print(["label"]) # (n,)
print(A["label"].shape, A["label"].dtype)
print(["meta"])  # structured array with label_source, label_pos, timestamp, sessionnum, blocknum, blockpos
print(A["meta"].shape, A["meta"].dtype)
print(["eeg"])   # float32 array (n, 128, 500)
print(A["eeg"].shape, A["eeg"].dtype)
print(["images"])
print(A["images"].shape, A["images"].dtype)


['label']
(32,) int64
['meta']
(32,) (numpy.record, [('label_source', 'O'), ('label_pos', '<i8'), ('timestamp', '<f8'), ('sessionnum', '<i8'), ('blocknum', '<i8'), ('blockpos', '<i8')])
['eeg']
(32, 128, 500) float32
['images']
(32, 28, 28) float32
